In [1]:
from hw2_helpers_games import Game, play_game, random_player, \
           h_alphabeta_search, cutoff_depth
from collections import defaultdict
import copy
import random

In [2]:
# Shapes

A, B, C, D = "\u25A0", "\u25B2", "\u25C6", "\u25CF"
print(A, B, C, D)

■ ▲ ◆ ●


In [3]:
class QBoard(defaultdict):
    """A board has the player to move, a cached utility value, 
    and a dict of {(x, y): (player, piece)} entries, where player is 'X' or 'O'."""
    
    empty = ('_','_')
    off = ('#','#')
    
    def __init__(self, width=4, height=4, to_move=None, **kwds):
        self.__dict__.update(width=width, height=height, to_move=to_move, **kwds)
        self.shapes = (A, B, C, D) 
        self.pieces = {'X': {A:0, B:0, C:0, D:0}, 'O': {A:0, B:0, C:0, D:0}}
        self.status = "Ready to play"
        
    def new(self, changes: dict, **kwds) -> 'Board':
        "Given a dict of {(x, y): (player, piece)} changes, return a new Board with the changes."
        board = QBoard(width=self.width, height=self.height, **kwds)
        board.update(self)
        board.update(changes)
        board.pieces = copy.deepcopy(self.pieces)
        for (player, piece) in changes.values():
            board.pieces[player][piece] += 1
        return board

    def __missing__(self, loc):
        x, y = loc
        if 0 <= x < self.width and 0 <= y < self.height:
            return self.empty
        else:
            return self.off
            
    def __hash__(self): 
        return hash(tuple(sorted(self.items()))) + hash(self.to_move)
    
    def __repr__(self):
        
        def row(y): return str(y) + '| ' +  \
            '  '.join(self[x, y][0]+'' + self[x, y][1] for x in range(self.width))
            
        return '    0   1   2   3\n' + \
            '\n'.join(map(row, range(self.height))) +  '\n'

In [4]:
def quad_pieces(square):
    x, y = square
    dx = 1 if (x == 0 or x == 2) else -1
    dy = 1 if (y == 0 or y == 2) else -1
    return [(x + dx, y), (x, y + dy), (x + dx, y + dy)]

In [5]:
def is_legal(board, player, square, piece):
    x, y = square
    other_player = 'O' if player == 'X' else 'X'
    for i in range(board.width):
        if board[i,y] == (other_player, piece):
            return False
    for j in range(board.height):
        if board[x,j] == (other_player, piece):
            return False
    for (i,j) in quad_pieces(square):
        if board[i,j] == (other_player, piece):
            return False
    return True

In [6]:
def full_board(board):
    return len(board) == 16

In [7]:
def check_win(game, board, square):
    all_shapes = sorted(list(board.shapes))
    x, y = square
    if sorted([board[i, y][1] for i in range(board.width)]) == all_shapes:
        board.status = f"Row {y} win"
        return True
    if sorted([board[x, j][1] for j in range(board.height)]) == all_shapes:
        board.status = f"Col {x} win"
        return True
    if sorted([board[i,j][1] for (i,j) in [square] + quad_pieces(square)]) == all_shapes:
        board.status = f"Quad {x},{y} win"
        return True
    if not full_board(board) and not game.actions(board):
        board.status = "No moves for opposite player, Win!"
        return True
    board.status =  "No winner"
    return False

In [8]:
class Quantik(Game):

    def __init__(self, width=4, height=4):
        
        self.squares = {(x, y) for x in range(width) for y in range(height)}
        self.initial = QBoard(to_move='X', utility=0)
        
    def actions(self, board):
        """Legal moves are any square not yet taken."""
        
        remaining_spaces = self.squares - set(board)
        player = board.to_move
        player_pieces = board.pieces[player]
        return [(location, piece) for location in remaining_spaces
                for piece in player_pieces 
                if player_pieces[piece] != 2 and is_legal(board, player, location, piece)]

    def result(self, board, action):
        """Place a marker for current player on square."""
        square, piece = action
        player = board.to_move
        board = board.new({square: (player, piece)}, to_move=('O' if player == 'X' else 'X'))
        
        win = check_win(self, board, square)
        board.utility = (0 if not win else +1 if player == 'X' else -1)
        return board

    def utility(self, board, player):
        """Return the value to player; 1 for win, -1 for loss, 0 otherwise."""
        
        return board.utility if player == 'X' else -board.utility

    def is_terminal(self, board):
        """A board is a terminal state if it is won or there are no empty squares."""
        
        return board.utility != 0 or len(self.squares) == len(board) or not self.actions(board)

    def display(self, board): 
        print(board)     


In [9]:
def print_game_summary(game, board):
   
    if result.utility == 1:
        print("X wins")
    elif result.utility == -1:
        print("O wins")
    else:
        print("Draw")

    print(board)
    print(board.status)

In [10]:
random.seed(6449)

q = Quantik()
result = play_game(q, 
          dict(X=random_player, 
               O=random_player), 
          verbose=True)

print_game_summary(q, result)

Step 1 After Player X move: ((2, 3), '■')
    0   1   2   3
0| __  __  __  __
1| __  __  __  __
2| __  __  __  __
3| __  __  X■  __

Step 2 After Player O move: ((1, 1), '●')
    0   1   2   3
0| __  __  __  __
1| __  O●  __  __
2| __  __  __  __
3| __  __  X■  __

Step 3 After Player X move: ((3, 0), '■')
    0   1   2   3
0| __  __  __  X■
1| __  O●  __  __
2| __  __  __  __
3| __  __  X■  __

Step 4 After Player O move: ((0, 0), '◆')
    0   1   2   3
0| O◆  __  __  X■
1| __  O●  __  __
2| __  __  __  __
3| __  __  X■  __

Step 5 After Player X move: ((2, 2), '●')
    0   1   2   3
0| O◆  __  __  X■
1| __  O●  __  __
2| __  __  X●  __
3| __  __  X■  __

Step 6 After Player O move: ((2, 0), '◆')
    0   1   2   3
0| O◆  __  O◆  X■
1| __  O●  __  __
2| __  __  X●  __
3| __  __  X■  __

Step 7 After Player X move: ((1, 0), '▲')
    0   1   2   3
0| O◆  X▲  O◆  X■
1| __  O●  __  __
2| __  __  X●  __
3| __  __  X■  __

Step 8 After Player O move: ((0, 2), '■')
    0   1   2   3
0| O◆  X▲

In [11]:
def first_player(game, state): 
    actions = sorted(game.actions(state))
    return actions[0]
 

def last_player(game, state): 
    actions = sorted(game.actions(state))
    return actions[-1]

In [12]:
q = Quantik()
result = play_game(q, 
          dict(X=first_player, 
               O=first_player), 
          verbose=True)
print_game_summary(q, result)


Step 1 After Player X move: ((0, 0), '■')
    0   1   2   3
0| X■  __  __  __
1| __  __  __  __
2| __  __  __  __
3| __  __  __  __

Step 2 After Player O move: ((0, 1), '▲')
    0   1   2   3
0| X■  __  __  __
1| O▲  __  __  __
2| __  __  __  __
3| __  __  __  __

Step 3 After Player X move: ((0, 2), '■')
    0   1   2   3
0| X■  __  __  __
1| O▲  __  __  __
2| X■  __  __  __
3| __  __  __  __

Step 4 After Player O move: ((0, 3), '▲')
    0   1   2   3
0| X■  __  __  __
1| O▲  __  __  __
2| X■  __  __  __
3| O▲  __  __  __

Step 5 After Player X move: ((1, 0), '◆')
    0   1   2   3
0| X■  X◆  __  __
1| O▲  __  __  __
2| X■  __  __  __
3| O▲  __  __  __

Step 6 After Player O move: ((1, 1), '●')
    0   1   2   3
0| X■  X◆  __  __
1| O▲  O●  __  __
2| X■  __  __  __
3| O▲  __  __  __

O wins
    0   1   2   3
0| X■  X◆  __  __
1| O▲  O●  __  __
2| X■  __  __  __
3| O▲  __  __  __

Quad 1,1 win


In [13]:
q = Quantik()
result = play_game(q, 
          dict(X=first_player, 
               O=last_player), 
          verbose=True)

print_game_summary(q, result)

Step 1 After Player X move: ((0, 0), '■')
    0   1   2   3
0| X■  __  __  __
1| __  __  __  __
2| __  __  __  __
3| __  __  __  __

Step 2 After Player O move: ((3, 3), '●')
    0   1   2   3
0| X■  __  __  __
1| __  __  __  __
2| __  __  __  __
3| __  __  __  O●

Step 3 After Player X move: ((0, 1), '■')
    0   1   2   3
0| X■  __  __  __
1| X■  __  __  __
2| __  __  __  __
3| __  __  __  O●

Step 4 After Player O move: ((3, 2), '●')
    0   1   2   3
0| X■  __  __  __
1| X■  __  __  __
2| __  __  __  O●
3| __  __  __  O●

Step 5 After Player X move: ((0, 2), '▲')
    0   1   2   3
0| X■  __  __  __
1| X■  __  __  __
2| X▲  __  __  O●
3| __  __  __  O●

Step 6 After Player O move: ((3, 1), '◆')
    0   1   2   3
0| X■  __  __  __
1| X■  __  __  O◆
2| X▲  __  __  O●
3| __  __  __  O●

Step 7 After Player X move: ((0, 3), '▲')
    0   1   2   3
0| X■  __  __  __
1| X■  __  __  O◆
2| X▲  __  __  O●
3| X▲  __  __  O●

Step 8 After Player O move: ((3, 0), '◆')
    0   1   2   3
0| X■  __